# Instituto Superior de Engenharia do Porto (ISEP)
## Licenciatura em Engenharia Física Aplicada (LEFA)
### Mecânica e Relatividade (MECREL) — 2026/2027
---
# Notebook TP04: Dinâmica da Partícula I — Leis de Newton e Modelos de Atrito

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cristovao-sousa-dias/MECREL/blob/main/TP04_Dinamica_Atrito_Colab.ipynb)

**Docente:** Cristóvão de Sousa Dias (`css@isep.ipp.pt`)  
**Objetivo Pedagógico:**
1. Explorar a resposta mecânica de um sistema sujeito a atrito seco de Coulomb (transição estático $\to$ cinético).
2. Visualizar dinamicamente a rampa de força aplicada, o limiar de escorregamento iminente e a queda abrupta da força de atrito.
3. **Auditar fisicamente** códigos gerados por Inteligência Artificial Generativa (LLMs) a partir do desafio da Parte E da Ficha TP04.

## 1. Setup & Bibliotecas Científicas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Estilo gráfico institucional
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.35
print("Ambiente configurado com sucesso!")

## 2. Modelo Físico de Referência: Transição de Atrito de Coulomb

O modelo de atrito seco de Coulomb sobre uma superfície horizontal plana com força exterior $F_{\mathrm{ext}}(t)$ estabelece que:

- **Regime Estático ($v = 0$):**  
  A força de atrito é **reativa e auto-ajustável**. Equilibra exatamente a força exterior aplicada até ao limiar crítico:
  $$f_s = -F_{\mathrm{ext}}, \quad \text{desde que } |F_{\mathrm{ext}}| \le f_{s,\max} = \mu_s N$$
  Neste regime, a aceleração é rigorosamente nula ($a = 0$).

- **Regime Cinético ($v \neq 0$ ou $|F_{\mathrm{ext}}| > \mu_s N$ a partir do repouso):**  
  As microssoldaduras são cizalhadas e o atrito passa a opor-se ao sentido do movimento relativo:
  $$f_k = -\operatorname{sign}(v)\,\mu_k N$$
  A aceleração resultante é dada pela 2ª Lei de Newton: $a = \frac{F_{\mathrm{ext}} + f_k}{m}$.

In [ ]:
def simular_bloco_rampa(m=5.0, mu_s=0.50, mu_k=0.35, t_max=10.0, dt=0.001, taxa_forca=5.0):
    """
    Simula um bloco sujeito a uma força exterior linearmente crescente F_ext(t) = taxa_forca * t.
    Implementa a lógica rigorosa de transição estático-cinético de Coulomb.
    """
    g = 9.8
    N = m * g
    f_s_max = mu_s * N
    
    n_steps = int(t_max / dt)
    t = np.linspace(0, t_max, n_steps)
    
    F_ext = taxa_forca * t
    x = np.zeros(n_steps)
    v = np.zeros(n_steps)
    a = np.zeros(n_steps)
    f_atrito = np.zeros(n_steps)
    
    v_tol = 1e-4  # tolerância numérica para repouso
    
    for i in range(n_steps - 1):
        # Verificação do regime
        if abs(v[i]) < v_tol:
            # Bloco praticamente em repouso
            if abs(F_ext[i]) <= f_s_max:
                # Regime estático: atrito equilibra a força aplicada
                f_atrito[i] = -F_ext[i]
                a[i] = 0.0
                v[i+1] = 0.0
            else:
                # Escorregamento iminente ultrapassado: transição para cinético
                f_atrito[i] = -np.sign(F_ext[i]) * mu_k * N
                a[i] = (F_ext[i] + f_atrito[i]) / m
                v[i+1] = v[i] + a[i] * dt
        else:
            # Regime cinético em movimento
            f_atrito[i] = -np.sign(v[i]) * mu_k * N
            a[i] = (F_ext[i] + f_atrito[i]) / m
            v[i+1] = v[i] + a[i] * dt
            
        x[i+1] = x[i] + v[i] * dt + 0.5 * a[i] * dt**2
        
    f_atrito[-1] = f_atrito[-2]
    a[-1] = a[-2]
    
    return t, F_ext, f_atrito, a, v, x

# Executar simulação de demonstração
t, F_ext, f_atrito, a, v, x = simular_bloco_rampa(m=5.0, mu_s=0.50, mu_k=0.35, t_max=8.0)

# Visualização dos resultados
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

ax1.plot(t, F_ext, label=r'Força Externa Aplicada $F_{\mathrm{ext}}(t)$', color='#0077B6', lw=2)
ax1.plot(t, -f_atrito, label=r'Magnitude do Atrito $|f_a(t)|$', color='#DC2626', lw=2.5)
ax1.axhline(0.50 * 5.0 * 9.8, color='#B45309', ls='--', label=r'Limiar Estático Máximo $f_{s,\max} = \mu_s N$')
ax1.axhline(0.35 * 5.0 * 9.8, color='#047857', ls=':', label=r'Patamar Cinético $f_k = \mu_k N$')
ax1.set_ylabel('Força [N]')
ax1.set_title('Dinâmica de Coulomb: Resposta Temporal e Queda na Transição Estático-Cinético')
ax1.legend(loc='upper left')

ax2.plot(t, a, label=r'Aceleração $a(t)$ [m/s$^2$]', color='#0284C7', lw=2)
ax2.plot(t, v, label=r'Velocidade $v(t)$ [m/s]', color='#10B981', lw=2)
ax2.set_xlabel('Tempo $t$ [s]')
ax2.set_ylabel('Cinemática')
ax2.legend(loc='upper left')

plt.tight_layout()
plt.show()

---
## 3. Desafio GenAI — Modelação e Auditoria Física de Códigos de IA

> ### 📌 Instruções para o Estudante:
> 1. Aceda à **Parte E da Ficha TP04** e copie o **Prompt Inicial de Teste** para o ChatGPT, Claude ou Gemini.
> 2. Cole a função gerada pelo seu assistente de IA na célula abaixo e execute o teste para os valores especificados:
>    - Massa: $m = 10\text{ kg}$
>    - Coeficientes: $\mu_s = 0.50$, $\mu_k = 0.30$
>    - Força aplicada: $F_{\mathrm{ext}} = 20\text{ N}$
>    - Partida do repouso: $v = 0$
> 3. **Analise o resultado obtido.** O comportamento físico observado na execução está de acordo com os princípios da Mecânica de Newton?
> 4. Responda às questões da ficha e discuta com o docente e a turma as condições necessárias para um código fisicamente rigoroso.

In [ ]:
# ==============================================================================
# ESPAÇO PARA O ALUNO: CÓDIGO GERADO PELO ASSISTENTE DE IA
# ==============================================================================

# TODO: Cole aqui a função gerada pelo ChatGPT / Gemini / Claude
# e execute o teste para m=10, F_ext=20, mu_s=0.5, mu_k=0.3, v=0



## 4. Implementação do Prompt Auditado & Validação

Após a discussão conceitual e a análise física em sala de aula, introduza no seu prompt os refinamentos necessários e teste na célula seguinte a versão final da função.

In [ ]:
# ==============================================================================
# ESPAÇO PARA O ALUNO: VERSÃO AUDITADA DA FUNÇÃO
# ==============================================================================

# TODO: Cole aqui a função após refinamento do prompt e valide o seu comportamento.

